# Evaluation of Segmented Linear Regression

> **Only the profiles: 'LRS70D_YSI_20230822' 'LRS75D_YSI_20230819'**

---

### Import Libraries

In [ ]:
import sys
import os
root = os.path.abspath('../..')  
sys.path.append(root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from piecewise_regression import r_squared_calc


from modules import load, plots, analysis, utils

# styles
plt.style.use('seaborn-v0_8-white')

---

### Load data

In [ ]:
name_70 = 'LRS70D_YSI_20230822'  
name_75 = 'LRS75D_YSI_20230819'

In [ ]:
path_json_70 = f'{root}/data/results/{name_70}_results.json'
path_json_75 = f'{root}/data/results/{name_75}_results.json'

df_70 = load.load_data(filepath=path_json_70, json=True)
df_75 = load.load_data(filepath=path_json_75, json=True)

In [ ]:
path_processed_70 = f'{root}/data/processed/{name_70}_processed.csv'
path_processed_75 = f'{root}/data/processed/{name_75}_processed.csv'

x_processed_70, y_processed_70 = load.load_data(filepath=path_processed_70, 
                            x_col='Vertical Position [m]',
                            y_col='Corrected sp Cond [uS/cm]'
                            )

x_processed_75, y_processed_75 = load.load_data(filepath=path_processed_75,
                            x_col='Vertical Position [m]',
                            y_col='Corrected sp Cond [uS/cm]'
                            )

In [ ]:
df_75

---

### Optimal `n_breakpoint`

In [ ]:
trial_70 = analysis.select_best_trial(path_json_70)
trial_select_70 = df_70[trial_70[0]]
N_BREAKPOINT_70 = df_70.loc['best_n_breakpoint_bic'].mode().iloc[0] # alternative, select 'best_n_breakpoint_rss'


trial_75 = analysis.select_best_trial(path_json_75)
trial_select_75 = df_75[trial_75[0]]
N_BREAKPOINT_75 = df_75.loc['best_n_breakpoint_bic'].mode().iloc[0] # alternative, select 'best_n_breakpoint_rss'

In [ ]:
# Elbow plot 70
x_values = np.array(list(trial_select_70['df']['n_breakpoints'].values()))
y_values = np.array(list(trial_select_70['df']['bic'].values()))
secondary_x = np.array(list(trial_select_70['df']['n_breakpoints'].values()))
secondary_y = np.array(list(trial_select_70['df']['rss'].values()))

plots.plot_data(
    x_values=x_values,
    y_values=y_values,
    plot_mode='lines+markers',
    x_axis_label="Number Breakpoints",
    y_axis_label="BIC",
    secondary_x=secondary_x,
    secondary_y=secondary_y,
    use_secondary_axis=True,
    y2_axis_label="RSS",
    trace_names=['BIC', 'RSS'],
    title=f"Elbow Plot: <b>{name_70}<b>",
)

# Elbow plot 75
x_values = np.array(list(trial_select_75['df']['n_breakpoints'].values()))
y_values = np.array(list(trial_select_75['df']['bic'].values()))
secondary_x = np.array(list(trial_select_75['df']['n_breakpoints'].values()))
secondary_y = np.array(list(trial_select_75['df']['rss'].values()))

plots.plot_data(
    x_values=x_values,
    y_values=y_values,
    plot_mode='lines+markers',
    x_axis_label="Number Breakpoints",
    y_axis_label="BIC",
    secondary_x=secondary_x,
    secondary_y=secondary_y,
    use_secondary_axis=True,
    y2_axis_label="RSS",
    trace_names=['BIC', 'RSS'],
    title=f"Elbow Plot: <b>{name_75}<b>",
)

---

### Evaluation

In [ ]:
# Params 70
params_70 = utils.get_breakpoint_data(trial_select_70['df'], N_BREAKPOINT_70)

# Params 75
params_75 = utils.get_breakpoint_data(trial_select_75['df'], N_BREAKPOINT_75)


In [ ]:
# Model 70
model_70 = utils.rebuild_model(x_processed_70,y_processed_70,params_70, tolerance=1e-2, min_distance=0.001)

# Model 75
model_75 = utils.rebuild_model(x_processed_75,y_processed_75,params_75, tolerance=1e-6, min_distance=0.008)

In [ ]:
# Globals 70
RSS_70, TSS_70, R2_70, R2_ajus_70 = r_squared_calc.get_r_squared(y_processed_70, 
                                                    model_70.predict(x_processed_70), 
                                                    len(params_70))

# Globals 75
RSS_75, TSS_75, R2_75, R2_ajus_75 = r_squared_calc.get_r_squared(y_processed_75, 
                                                    model_75.predict(x_processed_75), 
                                                    len(params_75))

print("RSS_70: ", RSS_70)
print("TSS_70: ", TSS_70)
print("R2_70: ", R2_70)
print("R2_ajus_70: ", R2_ajus_70)

print("RSS_75: ", RSS_75)
print("TSS_75: ", TSS_75)
print("R2_75: ", R2_75)
print("R2_ajus_75: ", R2_ajus_75)

In [ ]:
# Per segment 70
metric_per_segment_70 = analysis.calculate_metrics_per_segment(model_70)

# Per segment 75
metric_per_segment_75 = analysis.calculate_metrics_per_segment(model_75)

In [ ]:
# Breakpoints 70
breakpoints_70 = analysis.extract_breakpoints(model_70)

# Breakpoints 75
breakpoints_75 = analysis.extract_breakpoints(model_75)

In [ ]:
breakpoints_70

In [ ]:
breakpoints_75

---

### Final results


In [ ]:

# Visualizamos los datos procesados junto con los modelos obtenidos de 70
df_ms_70 = pd.DataFrame({'n_breakpoints': trial_select_70['df']['n_breakpoints'], 
                    'estimates': trial_select_70['df']['estimates']})

plots.interactive_segmented_regression(x=x_processed_70,
                                       y=y_processed_70, 
                                       df=df_ms_70, 
                                       title=name_70,
                                       breakpoints=N_BREAKPOINT_70)

# Visualizamos los datos procesados junto con los modelos obtenidos de 75
df_ms_75 = pd.DataFrame({'n_breakpoints': trial_select_75['df']['n_breakpoints'], 
                    'estimates': trial_select_75['df']['estimates']})

plots.interactive_segmented_regression(x=x_processed_75,
                                       y=y_processed_75, 
                                       df=df_ms_75, 
                                       title=name_75,
                                       breakpoints=N_BREAKPOINT_75)


#### Models per segment

In [ ]:
# Segments 70
segments_70 = utils.extract_segments(model_70)
segments_70

# Segments 75
segments_75 = utils.extract_segments(model_75)
segments_75

In [ ]:

# Plot 70
plots.plot_segments(segments_70, 
                    metric_per_segment_70, 
                    title=f"Segment-wise linear fitting: {name_70}")

# Plot 75
plots.plot_segments(segments_75, 
                    metric_per_segment_75, 
                    title=f"Segment-wise linear fitting: {name_75}")